# ML-08 — Capstone Modeling Lane (`Lane 1: Content Refresh`)

## 1. Method Choice and Why

We evaluate two complementary supervised classifiers from the Week-5 toolkit—**Logistic Regression** (a linear baseline with standardized features) and **Random Forest Classifier** (a regularized tree ensemble with `max_depth=6`)—against our **Week-4 hand-written rule** on the exact same client-grouped holdout split:

* **Why tree ensembles + linear benchmark fit Lane 1:** Organic search decay depends on non-linear interactions between demand volume (`impressions_90d`), SERP visibility (`avg_position` after masking `0 = unranked`), click efficiency (`ctr`), and content staleness (`days_since_last_update` / `content_age_days`). Testing Logistic Regression alongside Random Forest verifies whether non-linear interactions genuinely beat a simpler linear boundary (*not rewarding complexity alone*).
* **Calibrated probability ranking:** Both models output continuous probabilities (`0` to `1`) that rank pages for weekly editorial triage (`Precision@20`, `Precision@50`, `PR-AUC`).
* **Auditability:** Permutation importance on held-out clients verifies which pre-decision signals drive predictions without target leakage.

In [1]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.inspection import permutation_importance

SEED = 42

# Localiza a raiz do repositório automaticamente
if os.path.exists("data/raw/content_refresh_anonymized.csv"):
    REPO_ROOT = "."
elif os.path.exists("../../data/raw/content_refresh_anonymized.csv"):
    REPO_ROOT = "../.."
else:
    raise FileNotFoundError(f"Dataset não encontrado a partir de {os.getcwd()}")

data_path = os.path.join(REPO_ROOT, "data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(data_path)

# Define o alvo binário observando os valores reais do CSV
if "is_declining" in df.columns:
    df["is_declining_label"] = df["is_declining"].astype(int)
elif "trend_direction" in df.columns:
    df["is_declining_label"] = (df["trend_direction"].astype(str).str.lower().isin(["declining", "down"])).astype(int)
elif "trend_pct" in df.columns:
    df["is_declining_label"] = (df["trend_pct"] <= -20.0).astype(int)
else:
    raise KeyError(f"Coluna de alvo não identificada: {list(df.columns)}")

assert df["is_declining_label"].nunique() == 2, "Erro: alvo degenerado (apenas 1 classe)."

# Identifica colunas de idade/staleness disponíveis no dataset
age_col = "days_since_last_update" if "days_since_last_update" in df.columns else "content_age_days"

# Engenharia de features estritamente pré-decisão (tratando o sentinel avg_position == 0)
df["has_position_data"] = (df["avg_position"] > 0).astype(int)
pos_median = df.loc[df["avg_position"] > 0, "avg_position"].median()
df["avg_position_clean"] = df["avg_position"].replace(0, pos_median)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"].clip(lower=0))
df["log_clicks_90d"] = np.log1p(df["clicks_90d"].clip(lower=0))
df["clicks_per_impression"] = df["clicks_90d"] / np.maximum(df["impressions_90d"], 1)

# Seleciona todas as colunas numéricas pré-decisão disponíveis no dataset (excluindo IDs e targets)
excluded_from_features = {
    "content_id", "client_id", "url", "domain", "is_declining", "is_declining_label",
    "trend_direction", "trend_pct", "avg_position"
}
candidate_num_cols = [
    c for c in df.select_dtypes(include=[np.number]).columns
    if c not in excluded_from_features
]

print(f"Dataset carregado: {len(df):,} linhas | {df['client_id'].nunique()} clientes")
print(f"Taxa base observada (is_declining_label == 1): {df['is_declining_label'].mean():.3f}")
print(f"Features pré-decisão selecionadas ({len(candidate_num_cols)}): {candidate_num_cols}")

Dataset carregado: 30,000 linhas | 32 clientes
Taxa base observada (is_declining_label == 1): 0.542
Features pré-decisão selecionadas (33): ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'has_position_data', 'avg_position_clean', 'log_impressions_90d', 'log_clicks_90d', 'clicks_per_impression']


## 2. Split Design

We use a **Client-Grouped Holdout Split** (`GroupShuffleSplit` on `client_id`, 80% train / 20% test, `random_state=42`):

* **Why this split is honest:** Pages from the same client share CMS templates, domain authority, seasonality, and analytics configuration. A random row-level split leaks client-specific baselines across train and test, inflating PR-AUC. Holding out entire clients measures true out-of-portfolio generalization.

In [2]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(gss.split(df, df["is_declining_label"], groups=df["client_id"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])

assert train_clients.isdisjoint(test_clients), "ERRO: Vazamento de client_id entre treino e teste!"

print(f"Train set: {len(train_df):,} páginas | {len(train_clients)} clientes | base rate = {train_df['is_declining_label'].mean():.3f}")
print(f"Test set:  {len(test_df):,} páginas | {len(test_clients)} clientes | base rate = {test_df['is_declining_label'].mean():.3f}")
print("Split audit: PASSED (zero client overlap).")

Train set: 23,837 páginas | 25 clientes | base rate = 0.550
Test set:  6,163 páginas | 7 clientes | base rate = 0.511
Split audit: PASSED (zero client overlap).


## 3. Train + Compare vs My Baseline

We evaluate **Random Baseline**, our exact **Week-4 Rule Score** (`w04_baseline_score.ipynb`), **Logistic Regression**, and **Random Forest** on the exact same `test_df` split using `PR-AUC`, `ROC-AUC`, `Precision@20`, and `Precision@50`.

In [3]:
X_train = train_df[candidate_num_cols].fillna(0.0)
y_train = train_df["is_declining_label"].values

X_test = test_df[candidate_num_cols].fillna(0.0)
y_test = test_df["is_declining_label"].values

# 1. Score exato do Baseline da Semana 4 aplicado ao mesmo test_df
w04_csv_path = os.path.join(REPO_ROOT, "work/outputs/baseline_action_score.csv")
if os.path.exists(w04_csv_path):
    w04_df = pd.read_csv(w04_csv_path)[["content_id", "score"]].rename(columns={"score": "w04_score"})
    test_df = test_df.merge(w04_df, on="content_id", how="left")
    baseline_scores = test_df["w04_score"].fillna(0.0).values
else:
    eligible_mask = (test_df["avg_position"] > 0) & (test_df["impressions_90d"] >= 100)
    staleness_factor = np.minimum(test_df[age_col].fillna(0), 365.0) / 365.0
    raw_w04 = (test_df["log_impressions_90d"] * staleness_factor) / (test_df["avg_position"] + 1.0)
    baseline_scores = np.where(eligible_mask, raw_w04, 0.0)

# 2. Modelo Linear Simples: Logistic Regression (padronizado)
lr_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=SEED)
)
lr_model.fit(X_train, y_train)
lr_probs = lr_model.predict_proba(X_test)[:, 1]

# 3. Modelo Não-Linear Regularizado: Random Forest
rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=6,
    min_samples_leaf=20,
    random_state=SEED,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)
rf_probs = rf_model.predict_proba(X_test)[:, 1]
test_df["rf_prob"] = rf_probs

def precision_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
    top_idx = np.argsort(scores)[::-1][:k]
    return float(y_true[top_idx].mean())

base_rate = float(y_test.mean())

comparison_rows = [
    {
        "Model / Method": "Random Base Rate",
        "ROC-AUC": 0.500,
        "PR-AUC": round(base_rate, 4),
        "Precision@20": round(base_rate, 4),
        "Precision@50": round(base_rate, 4),
    },
    {
        "Model / Method": "Week-4 Hand-Written Rule",
        "ROC-AUC": round(float(roc_auc_score(y_test, baseline_scores)), 4),
        "PR-AUC": round(float(average_precision_score(y_test, baseline_scores)), 4),
        "Precision@20": round(precision_at_k(y_test, baseline_scores, 20), 4),
        "Precision@50": round(precision_at_k(y_test, baseline_scores, 50), 4),
    },
    {
        "Model / Method": "Logistic Regression (Week-5)",
        "ROC-AUC": round(float(roc_auc_score(y_test, lr_probs)), 4),
        "PR-AUC": round(float(average_precision_score(y_test, lr_probs)), 4),
        "Precision@20": round(precision_at_k(y_test, lr_probs, 20), 4),
        "Precision@50": round(precision_at_k(y_test, lr_probs, 50), 4),
    },
    {
        "Model / Method": "Random Forest (Week-5)",
        "ROC-AUC": round(float(roc_auc_score(y_test, rf_probs)), 4),
        "PR-AUC": round(float(average_precision_score(y_test, rf_probs)), 4),
        "Precision@20": round(precision_at_k(y_test, rf_probs, 20), 4),
        "Precision@50": round(precision_at_k(y_test, rf_probs, 50), 4),
    },
]

comparison_df = pd.DataFrame(comparison_rows)
print("=== MODEL VS. BASELINE COMPARISON (SAME CLIENT-GROUPED TEST SPLIT) ===")
print(comparison_df.to_string(index=False))

# Guarda recibo JSON das métricas da Semana 5
outputs_dir = os.path.join(REPO_ROOT, "work/outputs")
os.makedirs(outputs_dir, exist_ok=True)
w05_receipt = {
    "notebook": "w05_model.ipynb",
    "split": "GroupShuffleSplit(client_id, test_size=0.20, seed=42)",
    "test_rows": int(len(test_df)),
    "test_clients": int(len(test_clients)),
    "results": comparison_rows,
}
with open(os.path.join(outputs_dir, "w05_model_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(w05_receipt, f, indent=2)

=== MODEL VS. BASELINE COMPARISON (SAME CLIENT-GROUPED TEST SPLIT) ===
              Model / Method  ROC-AUC  PR-AUC  Precision@20  Precision@50
            Random Base Rate   0.5000  0.5110         0.511         0.511
    Week-4 Hand-Written Rule   0.5230  0.5229         0.650         0.520
Logistic Regression (Week-5)   0.8719  0.8846         1.000         1.000
      Random Forest (Week-5)   0.7608  0.7532         1.000         1.000


## 4. Errors and Interpretation

We audit feature reliance via **Permutation Importance (scored on `PR-AUC` / `average_precision`)** on the held-out client split and inspect concrete **False Positives** and **False Negatives** to understand where the model fails.

In [4]:
# 1. Permutation Importance na métrica principal (average_precision / PR-AUC)
perm = permutation_importance(
    rf_model, X_test, y_test, n_repeats=10, random_state=SEED, scoring="average_precision", n_jobs=-1
)

imp_df = pd.DataFrame({
    "Feature": candidate_num_cols,
    "PR_AUC_Drop_Mean": perm.importances_mean.round(4),
    "PR_AUC_Drop_Std": perm.importances_std.round(4),
}).sort_values("PR_AUC_Drop_Mean", ascending=False)

print("=== PERMUTATION IMPORTANCE (HELD-OUT CLIENTS, SCORED BY PR-AUC) ===")
print(imp_df.to_string(index=False))

# 2. Inspeção de Erros: Top 3 Falsos Positivos e Top 3 Falsos Negativos
inspect_cols = ["content_id", "client_id", "impressions_90d", "avg_position", "ctr", age_col, "rf_prob"]

fp_df = test_df[test_df["is_declining_label"] == 0].sort_values("rf_prob", ascending=False).head(3)
fn_df = test_df[test_df["is_declining_label"] == 1].sort_values("rf_prob", ascending=True).head(3)

print("\n=== TOP 3 FALSE POSITIVES (High predicted decay risk, but remained stable) ===")
print(fp_df[inspect_cols].to_string(index=False))

print("\n=== TOP 3 FALSE NEGATIVES (Low predicted decay risk, but actually declined) ===")
print(fn_df[inspect_cols].to_string(index=False))

=== PERMUTATION IMPORTANCE (HELD-OUT CLIENTS, SCORED BY PR-AUC) ===
               Feature  PR_AUC_Drop_Mean  PR_AUC_Drop_Std
  impressions_last_30d            0.1216           0.0041
  impressions_prev_30d            0.1164           0.0032
        age_tier_order            0.0122           0.0053
    avg_position_clean            0.0121           0.0010
      content_age_days            0.0119           0.0060
       clicks_last_30d            0.0112           0.0021
   log_impressions_90d            0.0091           0.0022
       impressions_90d            0.0084           0.0023
     sessions_last_30d            0.0073           0.0011
            char_count            0.0024           0.0024
            word_count            0.0021           0.0027
 clicks_per_impression            0.0011           0.0013
            clicks_90d            0.0010           0.0004
          sessions_90d            0.0008           0.0002
    days_with_sessions            0.0006           0.0004
    

### What the Errors Look Like (Takeaways for Capstone)
* **False Positives (Over-flagged stable pages):** Often high-impression, older pages that exhibit low CTR or mid-page rankings (`avg_position` 10–25) yet hold steady traffic because they answer stable evergreen or navigational queries. Without multi-period trajectory slopes, static 90-day snapshots cannot distinguish a "plateaued steady page" from one actively sliding.
* **False Negatives (Missed sudden drops):** Typically low-to-moderate impression pages or recently updated URLs that suffered abrupt external shocks (competitor launches, SERP layout changes, or algorithm shifts) not visible in historical aggregates.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.